In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI



template_1 = ChatPromptTemplate.from_template(
    "请简短回答！我现在正在进行{programming_language}语言编程，我想完成一个{goal}，请用文字简要说明需要实现的类和功能以及容易出现逻辑错误的地方,并在最后生成简短的prompt指引另一个AI完成编写任务。"
)

prompt_1 = template_1.format(programming_language = "Python",goal = "贪吃蛇小游戏")

print(prompt_1)

deepseek = ChatOllama(
    model="deepseek-r1:8b"
)

parser = StrOutputParser()

load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")

if not gemini_api_key:
    raise ValueError("Can't find GEMINI_API_KEY")

gemini = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    google_api_key=gemini_api_key
)

chain = template_1|deepseek|parser|gemini

response = chain.stream({"programming_language":"python","goal":"贪吃蛇小游戏"})

for chunk in response:
    print(chunk.content,end="",flush=False)


Human: 请简短回答！我现在正在进行Python语言编程，我想完成一个贪吃蛇小游戏，请用文字简要说明需要实现的类和功能以及容易出现逻辑错误的地方,并在最后生成简短的prompt指引另一个AI完成编写任务。
好的，这是一个使用`pygame`库实现的完整贪吃蛇游戏，包含了您指南中提到的核心功能和注意事项。

```python
import pygame
import random
import sys # 用于sys.exit()

# --- 游戏常量定义 ---
SCREEN_WIDTH = 600
SCREEN_HEIGHT = 400
GRID_SIZE = 20  # 蛇身和食物的单元格大小
GRID_WIDTH = SCREEN_WIDTH // GRID_SIZE
GRID_HEIGHT = SCREEN_HEIGHT // GRID_SIZE
SNAKE_SPEED = 10  # 游戏帧率，决定蛇的移动速度

# --- 颜色定义 ---
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GREEN = (0, 255, 0)  # 蛇的颜色
RED = (255, 0, 0)    # 食物的颜色
BLUE = (0, 0, 255)   # 分数和文字的颜色

# --- 方向定义 (dx, dy) ---
UP = (0, -GRID_SIZE)
DOWN = (0, GRID_SIZE)
LEFT = (-GRID_SIZE, 0)
RIGHT = (GRID_SIZE, 0)

# --- Pygame 初始化 ---
pygame.init()
pygame.display.set_caption("贪吃蛇")
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()
font = pygame.font.Font(None, 36) # 默认字体，大小36

# --- Snake类 ---
class Snake:
    def __init__(self):
        # 蛇的身体坐标列表，每个元素是一个(x, y)元组
        # 初始长度为3，从屏幕